In [1]:
# ============================================================
# Imports
# ============================================================

import requests
import pandas as pd
from bs4 import BeautifulSoup
from google.cloud import bigquery
from datetime import datetime, timezone
from io import StringIO
import unicodedata
import re

In [2]:
# ============================================================
# Helper Functions
# ============================================================

def normalise_team_name(text):
    if pd.isna(text):
        return text

    text = unicodedata.normalize("NFKD", str(text))
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = re.sub(r"\[.*?\]", "", text)   # Remove Wikipedia references
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [3]:
# ============================================================
# Configuration
# ============================================================

PROJECT_ID = "pacey32-agency"
DATASET = "Team"
TABLE = "OrganizationDetail"

TEAMLIST_SQL = """
SELECT *
FROM `pacey32-agency.Team.TeamList`
ORDER BY fullName
"""

URL_ARENAS = "https://en.wikipedia.org/wiki/List_of_National_Hockey_League_arenas"
URL_COACHES = "https://en.wikipedia.org/wiki/List_of_NHL_head_coaches"
URL_GMS = "https://en.wikipedia.org/wiki/List_of_current_NHL_general_managers"
URL_OWNERS = "https://en.wikipedia.org/wiki/List_of_current_NHL_franchise_owners"
URL_AHL = "https://en.wikipedia.org/wiki/American_Hockey_League"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/139.0 Safari/537.36"
    )
}

SCRAPE_TS = datetime.now(timezone.utc)

In [4]:
# ============================================================
# Connect to BigQuery
# ============================================================

client = bigquery.Client(project=PROJECT_ID)

print(f"Connected to {PROJECT_ID}")

Connected to pacey32-agency


In [45]:
# ============================================================
# Read Team.TeamList
# ============================================================

team_df = client.query(TEAMLIST_SQL).to_dataframe()

team_df["join_team"] = team_df["fullName"].apply(normalise_team_name)

print(f"{len(team_df)} teams loaded")

#display(team_df.head())

32 teams loaded


In [35]:
# ============================================================
# Scrape NHL Arenas
# ============================================================

response = requests.get(URL_ARENAS, headers=HEADERS)
response.raise_for_status()

tables = pd.read_html(StringIO(response.text))

arena_df = None

for i, table in enumerate(tables):

    cols = [str(c) for c in table.columns]

    if (
        "Arena" in cols
        and "Capacity" in cols
        and "Opened" in cols
        and "Season of first NHL game" in cols
    ):
        arena_df = table.copy()
        print(f"Arena table found (table {i})")
        break

if arena_df is None:
    raise RuntimeError("Could not locate NHL arena table on Wikipedia.")

#display(arena_df.head())

Arena table found (table 0)


In [46]:
# ============================================================
# Clean Arena Data
# ============================================================

arena_df = arena_df.rename(columns={
    "Team": "fullName",
    "Arena": "arena_name",
    "Capacity": "arena_capacity",
    "Opened": "arena_opened",
    "Season of first NHL game": "arena_first_nhl_season"
})

arena_df = arena_df[[
    "fullName",
    "arena_name",
    "arena_capacity",
    "arena_opened",
    "arena_first_nhl_season"
]].copy()

# Remove reference markers like [1]
arena_df["fullName"] = arena_df["fullName"].str.replace(r"\[.*?\]", "", regex=True).str.strip()

arena_df["arena_name"] = arena_df["arena_name"].str.replace(r"\[.*?\]", "", regex=True).str.strip()

# Capacity is sometimes "18,347[2]"
arena_df["arena_capacity"] = (
    arena_df["arena_capacity"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace(r"\[.*?\]", "", regex=True)
)

arena_df["arena_capacity"] = pd.to_numeric(
    arena_df["arena_capacity"],
    errors="coerce"
).astype("Int64")

arena_df["source_url"] = URL_ARENAS
arena_df["last_updated"] = SCRAPE_TS

arena_df["join_team"] = arena_df["fullName"].apply(normalise_team_name)

#display(arena_df)

In [47]:
print(f"{len(arena_df)} arenas found")

missing = (
    team_df[["fullName"]]
        .merge(arena_df[["fullName"]], how="left", indicator=True)
)

display(
    missing[missing["_merge"] != "both"]
)

32 arenas found


,fullName,_merge
14,Montréal Canadiens,left_only


In [52]:
# ============================================================
# Scrape NHL Head Coaches
# ============================================================

response = requests.get(URL_COACHES, headers=HEADERS)
response.raise_for_status()

tables = pd.read_html(StringIO(response.text))

coach_df = None

for i, table in enumerate(tables):

    # Flatten MultiIndex columns if present
    if isinstance(table.columns, pd.MultiIndex):
        table.columns = [
            c[0] if c[0] == c[1] else "_".join([str(x) for x in c if x])
            for c in table.columns
        ]

    cols = table.columns.tolist()

    if (
        "Team" in cols
        and "Coach" in cols
        and any(col.startswith("Start date") for col in cols)
    ):
        coach_df = table.copy()
        print(f"Coach table found (table {i})")
        break

if coach_df is None:
    raise RuntimeError("Could not locate NHL Head Coach table.")

#display(coach_df.head())

Coach table found (table 1)


In [53]:
# ============================================================
# Clean Head Coaches
# ============================================================

coach_df = coach_df.rename(columns={
    "Team": "fullName",
    "Coach": "head_coach",
    "Start date[b]": "head_coach_since"
})

coach_df = coach_df[[
    "fullName",
    "head_coach",
    "head_coach_since"
]].copy()

for col in ["fullName", "head_coach"]:
    coach_df[col] = (
        coach_df[col]
        .astype(str)
        .str.replace(r"\[.*?\]", "", regex=True)
        .str.strip()
    )

coach_df["join_team"] = coach_df["fullName"].apply(normalise_team_name)

coach_df["source_url"] = URL_COACHES
coach_df["last_updated"] = SCRAPE_TS

#display(coach_df)

In [25]:
missing = (
    team_df[["fullName"]]
    .merge(
        coach_df[["fullName"]],
        on="fullName",
        how="left",
        indicator=True
    )
)

display(missing[missing["_merge"] != "both"])

,fullName,_merge


In [38]:
# ============================================================
# Scrape NHL General Managers
# ============================================================

response = requests.get(URL_GMS, headers=HEADERS)
response.raise_for_status()

tables = pd.read_html(StringIO(response.text))

gm_df = None

for i, table in enumerate(tables):

    # Flatten MultiIndex columns
    if isinstance(table.columns, pd.MultiIndex):
        table.columns = [
            c[0] if c[0] == c[1] else "_".join(str(x) for x in c if x)
            for c in table.columns
        ]

    cols = table.columns.tolist()

    if (
        "Team" in cols
        and "General manager" in cols
    ):
        gm_df = table.copy()
        print(f"GM table found (table {i})")
        break

if gm_df is None:
    raise RuntimeError("Could not locate GM table.")

#display(gm_df.head())

GM table found (table 0)


In [39]:
# ============================================================
# Clean General Managers
# ============================================================

gm_df = gm_df.rename(columns={
    "Team": "fullName",
    "General manager": "general_manager",
    "Tenured since": "gm_since",
    "Pro career": "gm_playing_career"
})

gm_df = gm_df[[
    "fullName",
    "general_manager",
    "gm_since",
    "gm_playing_career"
]].copy()

for col in ["fullName", "general_manager"]:
    gm_df[col] = (
        gm_df[col]
        .astype(str)
        .str.replace(r"\[.*?\]", "", regex=True)
        .str.strip()
    )

gm_df["join_team"] = gm_df["fullName"].apply(normalise_team_name)

gm_df["source_url"] = URL_GMS
gm_df["last_updated"] = SCRAPE_TS

#display(gm_df)

In [29]:
# ============================================================
# QA - General Managers
# ============================================================

missing = (
    team_df[["fullName"]]
    .merge(
        gm_df[["fullName"]],
        on="fullName",
        how="left",
        indicator=True
    )
)

display(missing[missing["_merge"] != "both"])

,fullName,_merge


In [40]:
# ============================================================
# Scrape NHL Owners
# ============================================================

response = requests.get(URL_OWNERS, headers=HEADERS)
response.raise_for_status()

tables = pd.read_html(StringIO(response.text))

owner_df = None

for i, table in enumerate(tables):

    if isinstance(table.columns, pd.MultiIndex):
        table.columns = [
            c[0] if c[0] == c[1] else "_".join(str(x) for x in c if x)
            for c in table.columns
        ]

    cols = table.columns.tolist()

    if (
        "Franchise" in cols
        and "Principal owner(s)" in cols
    ):
        owner_df = table.copy()
        print(f"Owner table found (table {i})")
        break

if owner_df is None:
    raise RuntimeError("Could not locate Owner table.")

#display(owner_df.head())

Owner table found (table 0)


In [55]:
# ============================================================
# Clean Owners
# ============================================================

owner_df = owner_df.rename(columns={
    "Franchise": "fullName",
    "Principal owner(s)": "principal_owner",
    "Year purchased": "owner_since",
    "Purchase price (US$ millions)": "purchase_price_usd_m",
    "Adjusted price": "purchase_price_adjusted_usd_m"
})

owner_df = owner_df[[
    "fullName",
    "principal_owner",
    "owner_since",
    "purchase_price_usd_m",
    "purchase_price_adjusted_usd_m"
]].copy()

for col in ["fullName", "principal_owner"]:
    owner_df[col] = (
        owner_df[col]
        .astype(str)
        .str.replace(r"\[.*?\]", "", regex=True)
        .str.strip()
    )
    owner_df["owner_since"] = (
        owner_df["owner_since"]
        .astype(str)
        .str.replace(r"\[.*?\]", "", regex=True)
        .str.strip()
    )

owner_df["join_team"] = owner_df["fullName"].apply(normalise_team_name)

owner_df["source_url"] = URL_OWNERS
owner_df["last_updated"] = SCRAPE_TS

display(owner_df)

,fullName,principal_owner,owner_since,purchase_price_usd_m,purchase_price_adjusted_usd_m,join_team,source_url,last_updated
0,Anaheim Ducks,Henry Samueli,2005,70,$115 million,Anaheim Ducks,https://en.wikipedia.org/wiki/List_of_current_...,2026-08-04 21:05:55.171239+00:00
1,Boston Bruins,Jeremy Jacobs,1975,10,$59.8 million,Boston Bruins,https://en.wikipedia.org/wiki/List_of_current_...,2026-08-04 21:05:55.171239+00:00
2,Buffalo Sabres,Terry and Kim Pegula,2011,165,$236 million,Buffalo Sabres,https://en.wikipedia.org/wiki/List_of_current_...,2026-08-04 21:05:55.171239+00:00
3,Calgary Flames,N. Murray Edwards,1980,16,$62.5 million,Calgary Flames,https://en.wikipedia.org/wiki/List_of_current_...,2026-08-04 21:05:55.171239+00:00
4,Carolina Hurricanes,Tom Dundon,2018,420,$538 million,Carolina Hurricanes,https://en.wikipedia.org/wiki/List_of_current_...,2026-08-04 21:05:55.171239+00:00
5,Chicago Blackhawks,Danny Wirtz,1954,1,$12 million,Chicago Blackhawks,https://en.wikipedia.org/wiki/List_of_current_...,2026-08-04 21:05:55.171239+00:00
6,Colorado Avalanche,Ann Walton Kroenke,2000,202,$378 million,Colorado Avalanche,https://en.wikipedia.org/wiki/List_of_current_...,2026-08-04 21:05:55.171239+00:00
7,Columbus Blue Jackets,John P. McConnell,1997 & 2012,80 & 173,$160 million & $243 million,Columbus Blue Jackets,https://en.wikipedia.org/wiki/List_of_current_...,2026-08-04 21:05:55.171239+00:00
8,Dallas Stars,Tom Gaglardi,2011,240,$343 million,Dallas Stars,https://en.wikipedia.org/wiki/List_of_current_...,2026-08-04 21:05:55.171239+00:00
9,Detroit Red Wings,Marian Ilitch,1982,8.5,$28.4 million,Detroit Red Wings,https://en.wikipedia.org/wiki/List_of_current_...,2026-08-04 21:05:55.171239+00:00


In [41]:
# ============================================================
# Scrape AHL Teams
# ============================================================

response = requests.get(URL_AHL, headers=HEADERS)
response.raise_for_status()

tables = pd.read_html(StringIO(response.text))

ahl_df = None

for i, table in enumerate(tables):

    if isinstance(table.columns, pd.MultiIndex):
        table.columns = [
            c[0] if c[0] == c[1] else "_".join(str(x) for x in c if x)
            for c in table.columns
        ]

    cols = table.columns.tolist()

    if (
        "Team Name" in cols
        and "NHL affiliate" in cols
    ):
        ahl_df = table.copy()
        print(f"AHL table found (table {i})")
        break

if ahl_df is None:
    raise RuntimeError("Could not locate AHL table.")

display(ahl_df.head())

AHL table found (table 1)


,Conference,Division,Team Name,City,Arena,Capacity,Founded,Joined,Current city since,Head coach,NHL affiliate
0,Eastern,Atlantic,Charlotte Checkers,"Charlotte, North Carolina",Bojangles Coliseum,8600,1971[c 1],1971[c 1],2010,Geordie Kinnear,Florida Panthers
1,Eastern,Atlantic,Hartford Wolf Pack,"Hartford, Connecticut",PeoplesBank Arena,14750,1926[c 1],1936,1997,Jay Leach,New York Rangers
2,Eastern,Atlantic,Hershey Bears,"Hershey, Pennsylvania",Giant Center,10500,1938,1938,1938,Derek King,Washington Capitals
3,Eastern,Atlantic,Lehigh Valley Phantoms,"Allentown, Pennsylvania",PPL Center,8420,1996[c 1],1996[c 1],2014,John Snowden,Philadelphia Flyers
4,Eastern,Atlantic,Providence Bruins,"Providence, Rhode Island",Amica Mutual Pavilion,11273,1987[c 1],1987[c 1],1992,Trent Whitfield,Boston Bruins


In [43]:
# ============================================================
# Clean AHL Data
# ============================================================

ahl_df = ahl_df.rename(columns={
    "Team Name": "ahl_team",
    "City": "ahl_city",
    "Arena": "ahl_arena",
    "Capacity": "ahl_capacity",
    "Founded": "ahl_founded",
    "Joined": "ahl_joined",
    "Current city since": "ahl_current_city_since",
    "Head coach": "ahl_head_coach",
    "NHL affiliate": "fullName"
})

ahl_df = ahl_df[[
    "fullName",
    "ahl_team",
    "ahl_city",
    "ahl_arena",
    "ahl_capacity",
    "ahl_founded",
    "ahl_joined",
    "ahl_current_city_since",
    "ahl_head_coach"
]].copy()

for col in ["fullName", "ahl_team", "ahl_city", "ahl_arena", "ahl_head_coach","ahl_founded", "ahl_joined", "ahl_current_city_since"]:
    ahl_df[col] = (
        ahl_df[col]
        .astype(str)
        .str.replace(r"\[.*?\]", "", regex=True)
        .str.strip()
    )

ahl_df["join_team"] = ahl_df["fullName"].apply(normalise_team_name)

ahl_df["source_url"] = URL_AHL
ahl_df["last_updated"] = SCRAPE_TS

display(ahl_df)

,fullName,ahl_team,ahl_city,ahl_arena,ahl_capacity,ahl_founded,ahl_joined,ahl_current_city_since,ahl_head_coach,join_team,source_url,last_updated
0,Florida Panthers,Charlotte Checkers,"Charlotte, North Carolina",Bojangles Coliseum,8600,1971,1971,2010,Geordie Kinnear,Florida Panthers,https://en.wikipedia.org/wiki/American_Hockey_...,2026-08-04 21:05:55.171239+00:00
1,New York Rangers,Hartford Wolf Pack,"Hartford, Connecticut",PeoplesBank Arena,14750,1926,1936,1997,Jay Leach,New York Rangers,https://en.wikipedia.org/wiki/American_Hockey_...,2026-08-04 21:05:55.171239+00:00
2,Washington Capitals,Hershey Bears,"Hershey, Pennsylvania",Giant Center,10500,1938,1938,1938,Derek King,Washington Capitals,https://en.wikipedia.org/wiki/American_Hockey_...,2026-08-04 21:05:55.171239+00:00
3,Philadelphia Flyers,Lehigh Valley Phantoms,"Allentown, Pennsylvania",PPL Center,8420,1996,1996,2014,John Snowden,Philadelphia Flyers,https://en.wikipedia.org/wiki/American_Hockey_...,2026-08-04 21:05:55.171239+00:00
4,Boston Bruins,Providence Bruins,"Providence, Rhode Island",Amica Mutual Pavilion,11273,1987,1987,1992,Trent Whitfield,Boston Bruins,https://en.wikipedia.org/wiki/American_Hockey_...,2026-08-04 21:05:55.171239+00:00
5,St. Louis Blues,Springfield Thunderbirds,"Springfield, Massachusetts",MassMutual Center,6800,1975,1981,2016,Steve Ott,St. Louis Blues,https://en.wikipedia.org/wiki/American_Hockey_...,2026-08-04 21:05:55.171239+00:00
6,Pittsburgh Penguins,Wilkes-Barre/Scranton Penguins,"Wilkes-Barre Township, Pennsylvania",Mohegan Arena at Casey Plaza,8300,1981,1981,1999,Kirk MacDonald,Pittsburgh Penguins,https://en.wikipedia.org/wiki/American_Hockey_...,2026-08-04 21:05:55.171239+00:00
7,Ottawa Senators,Belleville Senators,"Belleville, Ontario",CAA Arena,4365,1972,1972,2017,Andrew Campbell,Ottawa Senators,https://en.wikipedia.org/wiki/American_Hockey_...,2026-08-04 21:05:55.171239+00:00
8,Columbus Blue Jackets,Cleveland Monsters,"Cleveland, Ohio",Rocket Arena,18926,1994,2001,2007,Nick Bootland,Columbus Blue Jackets,https://en.wikipedia.org/wiki/American_Hockey_...,2026-08-04 21:05:55.171239+00:00
9,New York Islanders,Hamilton Hammers,"Hamilton, Ontario",TD Coliseum,16386,2001,2001,2026,Jay McKee,New York Islanders,https://en.wikipedia.org/wiki/American_Hockey_...,2026-08-04 21:05:55.171239+00:00


In [56]:
# ============================================================
# Merge Organisation Data
# ============================================================

organisation_df = (
    team_df

    .merge(
        arena_df.drop(columns=["fullName"]),
        on="join_team",
        how="left"
    )

    .merge(
        coach_df.drop(columns=["fullName", "source_url", "last_updated"]),
        on="join_team",
        how="left"
    )

    .merge(
        gm_df.drop(columns=["fullName", "source_url", "last_updated"]),
        on="join_team",
        how="left"
    )

    .merge(
        owner_df.drop(columns=["fullName", "source_url", "last_updated"]),
        on="join_team",
        how="left"
    )

    .merge(
        ahl_df.drop(columns=["fullName", "source_url", "last_updated"]),
        on="join_team",
        how="left"
    )
)

display(organisation_df.head())

,id,franchiseid,fullName,tricode,venue,venueLocation,home_name,hometeamplacename,home_logo,teamName_default,...,purchase_price_usd_m,purchase_price_adjusted_usd_m,ahl_team,ahl_city,ahl_arena,ahl_capacity,ahl_founded,ahl_joined,ahl_current_city_since,ahl_head_coach
0,24,32.0,Anaheim Ducks,ANA,Honda Center,Anaheim,Ducks,Anaheim,https://assets.nhle.com/logos/nhl/svg/ANA_ligh...,Anaheim Ducks,...,70,$115 million,San Diego Gulls,"San Diego, California",Pechanga Arena,12920,2000,2000,2015,Dave Manson
1,6,6.0,Boston Bruins,BOS,TD Garden,Boston,Bruins,Boston,https://assets.nhle.com/logos/nhl/svg/BOS_ligh...,Boston Bruins,...,10,$59.8 million,Providence Bruins,"Providence, Rhode Island",Amica Mutual Pavilion,11273,1987,1987,1992,Trent Whitfield
2,7,19.0,Buffalo Sabres,BUF,KeyBank Center,Buffalo,Sabres,Buffalo,https://assets.nhle.com/logos/nhl/svg/BUF_ligh...,Buffalo Sabres,...,165,$236 million,Rochester Americans,"Rochester, New York",Blue Cross Arena,10662,1956,1956,1956,Michael Leone
3,20,21.0,Calgary Flames,CGY,Scotiabank Saddledome,Calgary,Flames,Calgary,https://assets.nhle.com/logos/nhl/svg/CGY_ligh...,Calgary Flames,...,16,$62.5 million,Calgary Wranglers,"Calgary, Alberta",Scotiabank Saddledome,19289,1977,1977,2022,Brett Sutter
4,12,26.0,Carolina Hurricanes,CAR,Lenovo Center,Raleigh,Hurricanes,Carolina,https://assets.nhle.com/logos/nhl/svg/CAR_ligh...,Carolina Hurricanes,...,420,$538 million,Chicago Wolves,"Rosemont, Illinois",Allstate Arena,16692,1994,2001,2001,Spiros Anastas
